In [3]:
# ============================================================
# eea_co2_pipeline.py
# ============================================================
# Extrait les données CO₂ EEA pour 4 groupes automobiles
# (Renault, Stellantis, Volkswagen, Toyota) de 2015 à 2025.
#
# Produit deux fichiers :
#   - eea_co2_detail.csv       : détail par motorisation
#   - eea_co2_summary.csv      : résumé par groupe/année
#   - co2_targets_ue.csv       : objectifs réglementaires UE 2015-2035
#
# Usage : python eea_co2_pipeline.py
#         (ou exécuter dans Jupyter)
# ============================================================

import requests
import pandas as pd

BASE = "https://discodata.eea.europa.eu/sql"


# ════════════════════════════════════════════════════════════
# OBJECTIFS RÉGLEMENTAIRES CO₂ — UE (valeurs légales officielles)
#
# Sources :
#   - 2015-2019 : Règlement (CE) n°443/2009 → 130 g/km NEDC
#   - 2020      : Règlement (UE) 2019/631   →  95 g/km NEDC
#   - 2021-2024 : Décision Commission 2021  → 115.1 g/km WLTP
#   - 2025-2029 : Règlement (UE) 2019/631   →  93.6 g/km WLTP (-15% vs 2021)
#   - 2030-2034 : Règlement (UE) 2023/851   →  49.5 g/km WLTP (-55% vs 2021)
#   - 2035+     : Règlement (UE) 2023/851   →   0.0 g/km (0 émission)
#
# Note : Ce sont les objectifs FLEET-WIDE (moyenne de l'UE).
# Les objectifs individuels par constructeur varient selon
# la masse moyenne de leur parc (ajustement ±quelques g/km).
# ════════════════════════════════════════════════════════════

CO2_TARGETS = {
    # Période NEDC
    2015: {"target_gkm": 130.0, "protocol": "NEDC",
           "base_legale": "Règlement (CE) 443/2009"},
    2016: {"target_gkm": 130.0, "protocol": "NEDC",
           "base_legale": "Règlement (CE) 443/2009"},
    2017: {"target_gkm": 130.0, "protocol": "NEDC",
           "base_legale": "Règlement (CE) 443/2009"},
    2018: {"target_gkm": 130.0, "protocol": "NEDC",
           "base_legale": "Règlement (CE) 443/2009"},
    2019: {"target_gkm": 130.0, "protocol": "NEDC",
           "base_legale": "Règlement (CE) 443/2009"},
    2020: {"target_gkm":  95.0, "protocol": "NEDC",
           "base_legale": "Règlement (UE) 2019/631"},

    # Période WLTP
    2021: {"target_gkm": 115.1, "protocol": "WLTP",
           "base_legale": "Décision Commission 2021 / Règlement (UE) 2019/631"},
    2022: {"target_gkm": 115.1, "protocol": "WLTP",
           "base_legale": "Décision Commission 2021 / Règlement (UE) 2019/631"},
    2023: {"target_gkm": 115.1, "protocol": "WLTP",
           "base_legale": "Décision Commission 2021 / Règlement (UE) 2019/631"},
    2024: {"target_gkm": 115.1, "protocol": "WLTP",
           "base_legale": "Décision Commission 2021 / Règlement (UE) 2019/631"},
    2025: {"target_gkm":  93.6, "protocol": "WLTP",
           "base_legale": "Règlement (UE) 2019/631 (-15% vs 2021)"},
    2026: {"target_gkm":  93.6, "protocol": "WLTP",
           "base_legale": "Règlement (UE) 2019/631 (-15% vs 2021)"},
    2027: {"target_gkm":  93.6, "protocol": "WLTP",
           "base_legale": "Règlement (UE) 2019/631 (-15% vs 2021)"},
    2028: {"target_gkm":  93.6, "protocol": "WLTP",
           "base_legale": "Règlement (UE) 2019/631 (-15% vs 2021)"},
    2029: {"target_gkm":  93.6, "protocol": "WLTP",
           "base_legale": "Règlement (UE) 2019/631 (-15% vs 2021)"},
    2030: {"target_gkm":  49.5, "protocol": "WLTP",
           "base_legale": "Règlement (UE) 2023/851 (-55% vs 2021)"},
    2031: {"target_gkm":  49.5, "protocol": "WLTP",
           "base_legale": "Règlement (UE) 2023/851 (-55% vs 2021)"},
    2032: {"target_gkm":  49.5, "protocol": "WLTP",
           "base_legale": "Règlement (UE) 2023/851 (-55% vs 2021)"},
    2033: {"target_gkm":  49.5, "protocol": "WLTP",
           "base_legale": "Règlement (UE) 2023/851 (-55% vs 2021)"},
    2034: {"target_gkm":  49.5, "protocol": "WLTP",
           "base_legale": "Règlement (UE) 2023/851 (-55% vs 2021)"},
    2035: {"target_gkm":   0.0, "protocol": "WLTP",
           "base_legale": "Règlement (UE) 2023/851 (100% réduction)"},
}


# ════════════════════════════════════════════════════════════
# CONFIG — Tables et colonnes EEA par année
# ════════════════════════════════════════════════════════════

ANNEE_CONFIG = {
    # 2015-2020 : table consolidée, NEDC, colonne CO₂ = "E (g/km)"
    **{a: {
        "table":    "[CO2Emission].[latest].[co2cars]",
        "status":   "F",
        "co2_col":  "E (g/km)",
        "protocol": "NEDC",
        "group_by": "Mp"
    } for a in range(2015, 2021)},

    # 2021 : table consolidée, WLTP
    2021: {
        "table":    "[CO2Emission].[latest].[co2cars]",
        "status":   "F",
        "co2_col":  "Ewltp (g/km)",
        "protocol": "WLTP",
        "group_by": "Mp"
    },

    # 2022-2024 : tables annuelles séparées, WLTP
    2022: {
        "table":    "[CO2Emission].[latest].[co2cars_2022Pv25]",
        "status":   None,
        "co2_col":  "Ewltp (g/km)",
        "protocol": "WLTP",
        "group_by": "Mp"
    },
    2023: {
        "table":    "[CO2Emission].[latest].[co2cars_2023Pv27]",
        "status":   None,
        "co2_col":  "Ewltp (g/km)",
        "protocol": "WLTP",
        "group_by": "Mp"
    },
    2024: {
        "table":    "[CO2Emission].[latest].[co2cars_2024Pv29]",
        "status":   None,
        "co2_col":  "Ewltp (g/km)",
        "protocol": "WLTP",
        "group_by": "Mp"
    },

    # 2025 : table annuelle, WLTP, grouper par Mh (Mp est vide)
    2025: {
        "table":    "[CO2Emission].[latest].[co2cars_2025Pv31]",
        "status":   None,
        "co2_col":  "Ewltp (g/km)",
        "protocol": "WLTP",
        "group_by": "Mh"
    },
}


# ════════════════════════════════════════════════════════════
# FONCTIONS DE MAPPING
# ════════════════════════════════════════════════════════════

def assign_group_mp(mp):
    """Mappe un nom de pool EEA (Mp) vers l'un de nos 4 groupes."""
    if not mp:
        return None
    mp_upper = str(mp).upper()
    if "RENAULT" in mp_upper:
        return "Renault Group"
    if any(k in mp_upper for k in ["STELLANTIS", "PSA", "FCA"]):
        return "Stellantis"
    if any(k in mp_upper for k in ["VOLKSW", "VW-", "VW "]):
        return "Volkswagen Group"
    if any(k in mp_upper for k in ["TOYOTA", "SUBARU", "MAZDA", "DAIHATSU"]):
        return "Toyota Group"
    return None


MH_MAPPING_2025 = {
    "RENAULT":                  "Renault Group",
    "DACIA":                    "Renault Group",
    "RENAULT TRUCKS":           None,
    "VOLKSWAGEN":               "Volkswagen Group",
    "SKODA":                    "Volkswagen Group",
    "AUDI AG":                  "Volkswagen Group",
    "AUDI SPORT":               "Volkswagen Group",
    "AUDI HUNGARIA":            "Volkswagen Group",
    "SEAT":                     "Volkswagen Group",
    "CUPRA":                    "Volkswagen Group",
    "PORSCHE":                  "Volkswagen Group",
    "STELLANTIS AUTO":          "Stellantis",
    "STELLANTIS EUROPE":        "Stellantis",
    "TOYOTA MOTOR CORPORATION": "Toyota Group",
    "TOYOTA":                   "Toyota Group",
    "LEXUS":                    "Toyota Group",
}

def assign_group_mh(mh):
    """Mappe un nom de fabricant (Mh) vers l'un de nos 4 groupes (2025)."""
    if not mh:
        return None
    return MH_MAPPING_2025.get(str(mh).strip().upper(), None)


def classify_ft(ft):
    """Classifie le type de carburant en 4 catégories."""
    if not ft:
        return "Autre"
    ft_upper = str(ft).upper().strip()
    if ft_upper == "ELECTRIC":
        return "VE"
    if "ELECTRIC" in ft_upper and "/" in ft_upper:
        return "PHEV"
    if ft_upper in ["PETROL", "DIESEL", "LPG", "NG", "ETHANOL",
                    "NG-BIOMETHANE", "HYDROGEN", "PETROL HYBRID",
                    "DIESEL HYBRID", "HYBRID"]:
        return "Thermique"
    return "Autre"


# ════════════════════════════════════════════════════════════
# EXTRACTION POUR UNE ANNÉE
# ════════════════════════════════════════════════════════════

def extraire_eea_annee(annee):
    """
    Requête l'API EEA et retourne les données agrégées
    par groupe × motorisation pour l'année donnée.
    """
    cfg       = ANNEE_CONFIG[annee]
    group_col = cfg["group_by"]

    conditions = ["1=1"]
    if cfg["status"]:
        conditions.append(f"Year = {annee}")
        conditions.append(f"Status = '{cfg['status']}'")
    where_clause = " AND ".join(conditions)

    query = f"""
    SELECT
        [{group_col}]           as grouping_key,
        UPPER(Ft)               as Ft_norm,
        COUNT(*)                as n_vehicles,
        AVG(CASE WHEN [{cfg['co2_col']}] > 0
                 THEN CAST([{cfg['co2_col']}] as float)
            END)                as avg_co2
    FROM {cfg['table']}
    WHERE {where_clause}
    GROUP BY [{group_col}], UPPER(Ft)
    ORDER BY n_vehicles DESC
    """

    params   = {"query": query, "p": 1, "nrOfHits": 500}
    response = requests.get(BASE, params=params, timeout=120)
    data     = response.json()

    if not data.get("results"):
        print(f"  ❌ {annee} — aucun résultat : {data.get('message', '')}")
        return pd.DataFrame()

    df = pd.DataFrame(data["results"])
    df["n_vehicles"] = pd.to_numeric(df["n_vehicles"], errors="coerce")
    df["avg_co2"]    = pd.to_numeric(df["avg_co2"],    errors="coerce")

    if group_col == "Mp":
        df["group"] = df["grouping_key"].apply(assign_group_mp)
    else:
        df["group"] = df["grouping_key"].apply(assign_group_mh)

    df["motorisation"] = df["Ft_norm"].apply(classify_ft)
    df = df[df["group"].notna()].copy()

    if df.empty:
        print(f"  ⚠️  {annee} — aucun groupe cible trouvé")
        return pd.DataFrame()

    # Agrégation par groupe × motorisation
    df["sum_co2"] = df["avg_co2"] * df["n_vehicles"]

    result = df.groupby(["group", "motorisation"]).agg(
        n_vehicles=("n_vehicles", "sum"),
        sum_co2   =("sum_co2",    "sum")
    ).reset_index()

    result["avg_co2_gkm"] = (result["sum_co2"] / result["n_vehicles"]).round(2)
    result["year"]         = annee
    result["protocol"]     = cfg["protocol"]

    # Part de chaque motorisation au sein du groupe
    totaux = result.groupby("group")["n_vehicles"].sum().rename("total_group")
    result = result.merge(totaux, on="group")
    result["pct_motorisation"] = (
        result["n_vehicles"] / result["total_group"] * 100
    ).round(2)

    cols = ["year", "group", "protocol", "motorisation",
            "n_vehicles", "pct_motorisation", "avg_co2_gkm"]
    return result[cols]


# ════════════════════════════════════════════════════════════
# PIPELINE PRINCIPAL
# ════════════════════════════════════════════════════════════

print("Extraction données CO₂ EEA 2015-2025...")
print("=" * 60)

all_data = []

for annee in range(2015, 2026):
    print(f"\n🔄 {annee}")
    df = extraire_eea_annee(annee)
    if not df.empty:
        all_data.append(df)
        print(f"  ✅ {df['group'].nunique()} groupes | {df.shape[0]} lignes")

# ── Table détail ─────────────────────────────────────────────
df_detail = pd.concat(all_data, ignore_index=True)
df_detail = df_detail.sort_values(
    ["year", "group", "motorisation"]
).reset_index(drop=True)

# ── Table résumé (toutes motorisations agrégées) ──────────────
# Le gap vs target sera calculé dans Power BI via mesure DAX
df_summary = df_detail.groupby(["year", "group", "protocol"]).apply(
    lambda g: pd.Series({
        "n_vehicles_total":   g["n_vehicles"].sum(),
        "avg_co2_fleet_gkm":  round(
            (g["avg_co2_gkm"] * g["n_vehicles"]).sum()
            / g["n_vehicles"].sum(), 2
        ),
        "pct_ve":        g.loc[g["motorisation"] == "VE",
                               "pct_motorisation"].sum().round(2),
        "pct_phev":      g.loc[g["motorisation"] == "PHEV",
                               "pct_motorisation"].sum().round(2),
        "pct_thermique": g.loc[g["motorisation"] == "Thermique",
                               "pct_motorisation"].sum().round(2),
    })
).reset_index()

# ── Table objectifs réglementaires ───────────────────────────
df_targets = pd.DataFrame([
    {
        "year":         annee,
        "target_gkm":  v["target_gkm"],
        "protocol":    v["protocol"],
        "base_legale": v["base_legale"]
    }
    for annee, v in CO2_TARGETS.items()
]).sort_values("year").reset_index(drop=True)

# ── Sauvegardes ──────────────────────────────────────────────
df_detail.to_csv("data/processed/eea_co2_detail.csv",   index=False)
df_summary.to_csv("data/processed/eea_co2_summary.csv", index=False)
df_targets.to_csv("data/processed/co2_targets_ue.csv",  index=False)

print("\n" + "=" * 60)
print("✅ Fichiers sauvegardés dans data/processed/")
print(f"   eea_co2_detail.csv  : {df_detail.shape[0]} lignes "
      f"({df_detail['year'].min()}-{df_detail['year'].max()})")
print(f"   eea_co2_summary.csv : {df_summary.shape[0]} lignes")
print(f"   co2_targets_ue.csv  : {df_targets.shape[0]} lignes "
      f"(2015-2035)")

print("\n── Aperçu summary 2024 ──")
cols_aff = ["year", "group", "avg_co2_fleet_gkm",
            "pct_ve", "pct_phev", "pct_thermique"]
print(df_summary[df_summary["year"] == 2024][cols_aff].to_string(index=False))

print("\n── Objectifs réglementaires ──")
print(df_targets.to_string(index=False))

Extraction données CO₂ EEA 2015-2025...

🔄 2015
  ✅ 4 groupes | 12 lignes

🔄 2016
  ✅ 3 groupes | 7 lignes

🔄 2017
  ✅ 3 groupes | 10 lignes

🔄 2018
  ✅ 4 groupes | 12 lignes

🔄 2019
  ✅ 4 groupes | 13 lignes

🔄 2020
  ✅ 4 groupes | 14 lignes

🔄 2021
  ✅ 4 groupes | 12 lignes

🔄 2022
  ✅ 4 groupes | 15 lignes

🔄 2023
  ✅ 4 groupes | 14 lignes

🔄 2024
  ✅ 4 groupes | 13 lignes

🔄 2025
  ✅ 4 groupes | 15 lignes

✅ Fichiers sauvegardés dans data/processed/
   eea_co2_detail.csv  : 137 lignes (2015-2025)
   eea_co2_summary.csv : 42 lignes
   co2_targets_ue.csv  : 21 lignes (2015-2035)

── Aperçu summary 2024 ──
 year            group  avg_co2_fleet_gkm  pct_ve  pct_phev  pct_thermique
 2024    Renault Group             112.39    7.94      1.14          90.91
 2024       Stellantis             108.75   10.23      3.75          86.02
 2024     Toyota Group             109.35    2.26      4.52          93.19
 2024 Volkswagen Group             118.13   11.98      6.23          81.79

── Object

In [4]:
data = [
    {
        "plan":         "Renaulution",
        "date_annonce": "2021-01-14",
        "categorie":    "Électrification",
        "objectif":     "Nouveaux modèles VE lancés",
        "valeur_cible": "≥10 VE d'ici 2025",
        "horizon":      2025,
        "resultat":     "1.5 million hybrides vendus 2021-2025 / n°1 CO₂ généralistes",
        "source":       "Plan futuREady mars 2026 — renaultgroup.com"
    },
    {
        "plan":         "Renaulution",
        "date_annonce": "2021-01-14",
        "categorie":    "Environnement",
        "objectif":     "Neutralité carbone Europe",
        "valeur_cible": "0 g CO₂",
        "horizon":      2050,
        "resultat":     None,
        "source":       "Plan Renaulution — media.renaultgroup.com"
    },
    {
        "plan":         "Ampere",
        "date_annonce": "2023-11-01",
        "categorie":    "Électrification",
        "objectif":     "Production VE Renault (Ampere)",
        "valeur_cible": "1 million VE/an",
        "horizon":      2031,
        "resultat":     None,
        "source":       "Capital Markets Day Ampere — renaultgroup.com"
    },
    {
        "plan":         "Ampere",
        "date_annonce": "2023-11-01",
        "categorie":    "Électrification",
        "objectif":     "Modèles électriques Ampere en Europe",
        "valeur_cible": "6 modèles",
        "horizon":      2030,
        "resultat":     None,
        "source":       "Capital Markets Day Ampere — renaultgroup.com"
    },
    {
        "plan":         "futuREady",
        "date_annonce": "2026-03-10",
        "categorie":    "Volume",
        "objectif":     "Ventes annuelles marque Renault",
        "valeur_cible": "2 millions véhicules/an",
        "horizon":      2030,
        "resultat":     "1.628 million en 2025",
        "source":       "Plan futuREady — renaultgroup.com"
    },
    {
        "plan":         "futuREady",
        "date_annonce": "2026-03-10",
        "categorie":    "Électrification",
        "objectif":     "Nouveaux modèles électriques en Europe",
        "valeur_cible": "16 VE sur 22 modèles lancés",
        "horizon":      2030,
        "resultat":     None,
        "source":       "Plan futuREady — renaultgroup.com"
    },
    {
        "plan":         "futuREady",
        "date_annonce": "2026-03-10",
        "categorie":    "CO2",
        "objectif":     "Position CO₂ marque Renault en Europe",
        "valeur_cible": "Meilleure marque généraliste",
        "horizon":      2030,
        "resultat":     "Déjà n°1 généraliste CO₂ en 2025",
        "source":       "Plan futuREady — renaultgroup.com"
    },
]

df_ir = pd.DataFrame(data)
df_ir.to_csv("data/processed/renault_ir_objectifs.csv", index=False)
print(f"✅ {df_ir.shape[0]} objectifs sauvegardés")
print(df_ir[["plan","categorie","objectif","valeur_cible","horizon"]].to_string(index=False))

✅ 7 objectifs sauvegardés
       plan       categorie                               objectif                 valeur_cible  horizon
Renaulution Électrification             Nouveaux modèles VE lancés            ≥10 VE d'ici 2025     2025
Renaulution   Environnement              Neutralité carbone Europe                      0 g CO₂     2050
     Ampere Électrification         Production VE Renault (Ampere)              1 million VE/an     2031
     Ampere Électrification   Modèles électriques Ampere en Europe                    6 modèles     2030
  futuREady          Volume        Ventes annuelles marque Renault      2 millions véhicules/an     2030
  futuREady Électrification Nouveaux modèles électriques en Europe  16 VE sur 22 modèles lancés     2030
  futuREady             CO2  Position CO₂ marque Renault en Europe Meilleure marque généraliste     2030
